# CoWork Enterprise Demo — Evaluate, Go Live & Operate
## Notebook 05 — Evaluate, Go Live & Operate (Phase 5)

_Icons used throughout: 🛠️ Action  📌 Note  🔹 Info_

---

### What This Notebook Does:

1. 🛠️ **Evaluates** the agent — golden smoke tests + a scored **Agent GPA** run against a curated dataset
2. 🛠️ **Publishes** the agent to the shared **CoWork object** (the curation gate) and grants access
3. 🛠️ **Brands** the agent and the CoWork interface
4. 🛠️ Sets up **operations** (published check + health-check pattern)

---

### Why Evaluation Is the Go-Live Gate:

🔹 Never publish an agent a business user can reach without a scored evaluation behind it. The flow is **Evaluate (gate) → Go live (publish) → Operate (monitor)**.

🔹 **Agent GPA** scores the agent at each stage of its reasoning (Goal → Plan → Action), not just the final answer: `answer_correctness`, `logical_consistency`, and `tool_selection_accuracy`. Require a threshold (e.g. ≥ 0.85, 0 regressions) before publishing.

🔹 **Publishing is a curation gate:** agents are invisible in CoWork until an admin with `MODIFY` explicitly adds them to the singleton CoWork object. Only publish agents that passed the gate.

> **Documentation:** [Cortex Agent evaluations](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-agents-evaluations)

---

### Estimated Time: **25–30 minutes**

### Prerequisites:
- Notebook 04 complete (agent built; cost controls in place)

In [ ]:
%%sql -r set_context
USE ROLE COWORK_ENTERPRISE_DEMO_ADMIN;
USE DATABASE COWORK_ENTERPRISE_DEMO;
USE SCHEMA AGENTS;
USE WAREHOUSE COWORK_ENTERPRISE_DEMO_WH;


## Step 1: Evaluate — Golden Smoke Tests

🛠️ Quick, runnable checks for fast iteration. Cover a **mix** of question types — core analytics, tool routing, multi-tool, and out-of-scope refusal — so you catch different failure modes.

In [ ]:
%%sql -r golden_q1
SELECT SNOWFLAKE.CORTEX.DATA_AGENT_RUN(
  'COWORK_ENTERPRISE_DEMO.AGENTS.DEMO_AGENT',
  '{"messages": [{"role": "user", "content": [{"type": "text", "text": "Who are our top 5 clients by AUM?"}]}], "stream": false}'
) AS resp;


In [ ]:
%%sql -r golden_q2
SELECT SNOWFLAKE.CORTEX.DATA_AGENT_RUN(
  'COWORK_ENTERPRISE_DEMO.AGENTS.DEMO_AGENT',
  '{"messages": [{"role": "user", "content": [{"type": "text", "text": "Are there any compliance concerns I should know about?"}]}], "stream": false}'
) AS resp;


In [ ]:
%%sql -r golden_q3
SELECT SNOWFLAKE.CORTEX.DATA_AGENT_RUN(
  'COWORK_ENTERPRISE_DEMO.AGENTS.DEMO_AGENT',
  '{"messages": [{"role": "user", "content": [{"type": "text", "text": "Summarize recent trades over $1M and the rationale behind them."}]}], "stream": false}'
) AS resp;


In [ ]:
%%sql -r golden_q4
SELECT SNOWFLAKE.CORTEX.DATA_AGENT_RUN(
  'COWORK_ENTERPRISE_DEMO.AGENTS.DEMO_AGENT',
  '{"messages": [{"role": "user", "content": [{"type": "text", "text": "What is the weather in Sydney today?"}]}], "stream": false}'
) AS resp;


## Step 2: Evaluate — Agent GPA (scored gate)

🛠️ The golden smoke tests confirm the agent *answers*; Agent GPA **scores** it. Build a curated question bank with ground truth, register it as an evaluation dataset, then run `EXECUTE_AI_EVALUATION`.

📌 **Gate:** require a threshold (e.g. `answer_correctness >= 0.85`, 0 regressions) before publishing. Notebook 07 reuses this exact dataset for the continuous-improvement loop.

In [ ]:
%%sql -r create_eval_dataset
CREATE OR REPLACE TABLE COWORK_ENTERPRISE_DEMO.AGENTS.EVAL_QUESTIONS (
  INPUT_QUERY VARCHAR,
  GROUND_TRUTH VARIANT
);

INSERT INTO COWORK_ENTERPRISE_DEMO.AGENTS.EVAL_QUESTIONS (INPUT_QUERY, GROUND_TRUTH)
  SELECT 'Who are our top 5 clients by AUM?', PARSE_JSON('{"ground_truth_output": "Returns exactly five clients ranked by assets under management, largest first, with a value per client. Should read from the structured portfolio/client data, not research notes.", "ground_truth_invocations": [{"tool_name": "nexus_analytics"}]}')
UNION ALL
  SELECT 'What is our total technology sector exposure?', PARSE_JSON('{"ground_truth_output": "Gives a single aggregate exposure figure for the technology sector across positions, presented as a formatted currency amount. Derived from structured position data.", "ground_truth_invocations": [{"tool_name": "nexus_analytics"}]}')
UNION ALL
  SELECT 'Are there any compliance concerns I should know about?', PARSE_JSON('{"ground_truth_output": "Surfaces qualitative compliance flags/issues from the research and compliance notes and clearly identifies them as compliance concerns. Should draw on the research/search corpus, not fabricate numbers.", "ground_truth_invocations": [{"tool_name": "nexus_research"}]}')
UNION ALL
  SELECT 'Summarize recent trades over $1M and the rationale behind them.', PARSE_JSON('{"ground_truth_output": "Lists recent trades above $1M with amounts (from structured trade data) AND explains the rationale (from research notes), citing both sources. A complete answer uses both structured and unstructured data.", "ground_truth_invocations": [{"tool_name": "nexus_analytics"},{"tool_name": "nexus_research"}]}')
UNION ALL
  SELECT 'What''s the weather in Sydney today?', PARSE_JSON('{"ground_truth_output": "States that weather is outside the agent''s scope (portfolios, trades, research) and does not fabricate a forecast. A graceful refusal, not a hallucinated answer.", "ground_truth_invocations": []}')
UNION ALL
  SELECT 'Show AUM by region.', PARSE_JSON('{"ground_truth_output": "Returns assets under management broken down by client region, one row per region with a value, sourced from structured data.", "ground_truth_invocations": [{"tool_name": "nexus_analytics"}]}');


In [ ]:
%%sql -r register_eval_dataset
CALL SYSTEM$CREATE_EVALUATION_DATASET(
  'Cortex Agent',
  'COWORK_ENTERPRISE_DEMO.AGENTS.EVAL_QUESTIONS',
  'COWORK_ENTERPRISE_DEMO.AGENTS.DEMO_AGENT_EVALSET',
  OBJECT_CONSTRUCT('query_text', 'INPUT_QUERY', 'expected_tools', 'GROUND_TRUTH')
);


In [ ]:
%%sql -r create_eval_stage
CREATE FILE FORMAT IF NOT EXISTS COWORK_ENTERPRISE_DEMO.AGENTS.YAML_FF
  TYPE = 'CSV' FIELD_DELIMITER = NONE RECORD_DELIMITER = '\n'
  SKIP_HEADER = 0 FIELD_OPTIONALLY_ENCLOSED_BY = NONE ESCAPE_UNENCLOSED_FIELD = NONE;
CREATE STAGE IF NOT EXISTS COWORK_ENTERPRISE_DEMO.AGENTS.EVAL_CONFIG FILE_FORMAT = COWORK_ENTERPRISE_DEMO.AGENTS.YAML_FF;


### Run the scored evaluation
`EXECUTE_AI_EVALUATION` reads a config YAML **from a stage**. Save the YAML below as `agent_eval.yaml` in this workspace, upload it to the stage (Snowsight **Add Data -> Load files into a Stage**, or `PUT`/`COPY FILES`), then run the evaluation. **Easier path:** in the CoCo CLI, use the `cortex-agent` skill (`evaluate-cortex-agent`) to run the whole flow in one prompt.

```yaml
evaluation:
  agent_params:
    agent_name: "COWORK_ENTERPRISE_DEMO.AGENTS.DEMO_AGENT"
    agent_type: "CORTEX AGENT"
  run_params:
    label: "Go-live gate"
  source_metadata:
    type: "dataset"
    dataset_name: "COWORK_ENTERPRISE_DEMO.AGENTS.DEMO_AGENT_EVALSET"
metrics:
  - "answer_correctness"
  - "logical_consistency"
  - "tool_selection_accuracy"
```
```sql
-- After the YAML is on the stage:
CALL EXECUTE_AI_EVALUATION('START', OBJECT_CONSTRUCT('run_name', 'go_live_gate'),
  '@COWORK_ENTERPRISE_DEMO.AGENTS.EVAL_CONFIG/agent_eval.yaml');
-- Poll until COMPLETED:
CALL EXECUTE_AI_EVALUATION('STATUS', OBJECT_CONSTRUCT('run_name', 'go_live_gate'),
  '@COWORK_ENTERPRISE_DEMO.AGENTS.EVAL_CONFIG/agent_eval.yaml');
-- Read scores (gate on answer_correctness >= 0.85):
SELECT METRIC_NAME, AVG(EVAL_AGG_SCORE) AS AVG_SCORE
FROM TABLE(SNOWFLAKE.LOCAL.GET_AI_EVALUATION_DATA(
  'COWORK_ENTERPRISE_DEMO', 'AGENTS', 'DEMO_AGENT', 'CORTEX AGENT', 'go_live_gate'))
GROUP BY METRIC_NAME;
```
> Also review scores in Snowsight: **AI & ML -> Agents -> DEMO_AGENT -> Evaluations**.

## Step 3: Go Live — Publish to the CoWork Object

🛠️ The account already has a CoWork object (`SNOWFLAKE_INTELLIGENCE_OBJECT_DEFAULT`, a singleton). We **add our agent to it** (the curation gate) and grant the consumer role `USAGE` on the object. We never create or drop the shared object.

📌 Requires `MODIFY` on the object, so this uses ACCOUNTADMIN.

🔹 **Curation, and the Preview→GA change:** the CoWork object curates *which* agents appear in the dropdown. Since GA, **without** curation (or if an agent isn't added) users see **every** agent they have `USAGE` on — flexible but noisy. **With** curation, only agents you add show — your production-ready shortlist. (In Preview, an agent had to live in a specific schema to appear; that restriction is gone.)

🔗 **Deep links** reach a *non-curated* agent directly — useful to test before you curate: `https://ai.snowflake.com/{org-account}/#/agents/{database}.{schema}.{agent_name}`

💬 **For customers:** (1) there is **one** CoWork object per account — it's account-level, not per-database; (2) there's no per-role curated list — use **RBAC** (`USAGE ON AGENT`) to control who sees what, and the object to curate the shared shortlist.

In [ ]:
%%sql -r publish_agent
USE ROLE ACCOUNTADMIN;
ALTER SNOWFLAKE INTELLIGENCE SNOWFLAKE_INTELLIGENCE_OBJECT_DEFAULT ADD AGENT COWORK_ENTERPRISE_DEMO.AGENTS.DEMO_AGENT;
GRANT USAGE ON SNOWFLAKE INTELLIGENCE SNOWFLAKE_INTELLIGENCE_OBJECT_DEFAULT TO ROLE COWORK_ENTERPRISE_DEMO_SI_USER;


In [ ]:
%%sql -r brand_agent
ALTER AGENT COWORK_ENTERPRISE_DEMO.AGENTS.DEMO_AGENT SET
  PROFILE = '{"display_name": "Enterprise Demo Analyst", "avatar": "shield", "color": "blue"}';


## Step 4: Brand the CoWork Interface (account-level)

🛠️ The agent `PROFILE` above sets its display name/avatar/colour. The CoWork **object** carries the enterprise branding every user sees — brand name, welcome message, accent colour.

📌 This is account-level (the object is a singleton), so it **overwrites any prior branding**. Logos/icons use staged image paths; upload images to a stage or set them via AI & ML → Agents → Open settings.

In [ ]:
%%sql -r brand_cowork
USE ROLE ACCOUNTADMIN;
ALTER SNOWFLAKE INTELLIGENCE SNOWFLAKE_INTELLIGENCE_OBJECT_DEFAULT SET
  BRAND_NAME = 'Nexus Capital'
  WELCOME_MESSAGE = 'Welcome to Nexus Capital Intelligence. Ask about client portfolios, positions, trades, and market research.'
  ACCENT_COLOR_LIGHT = '#0B3D91'
  ACCENT_COLOR_DARK = '#5B9BD5';


## Restrict business users to CoWork (reference)
Confine non-technical users to the CoWork UI and ensure each has a default role + warehouse. Shown for reference - we do not lock a real user on a shared account.

```sql
ALTER USER <business_user> SET
  DEFAULT_ROLE = COWORK_ENTERPRISE_DEMO_SI_USER,
  DEFAULT_WAREHOUSE = COWORK_ENTERPRISE_DEMO_WH,
  ALLOWED_INTERFACES = (SNOWFLAKE_INTELLIGENCE);
-- IdP redirect so users never see a native login page:
ALTER ACCOUNT SET LOGIN_IDP_REDIRECT = (SNOWFLAKE_INTELLIGENCE = <security_integration>);
```

### 🔹 The Business-User Experience in CoWork

Once published, business users chat with the agent in CoWork — a different surface from the Agent Admin **Test** tab that you (the builder) use:

| | Agent Admin → Test | CoWork chat |
|---|---|---|
| Full agent spec visible / editable | Yes | No |
| Who uses it | Agent builders | Business users |
| MCP connectors | Configure (add/remove) | Connect & use (per-user OAuth) |
| Save / share / explore answers | No | **Yes** |
| Upload files for ad-hoc analysis | No | Yes (via + menu) |

**Both** surfaces show thinking steps, generated SQL, cited sources, inline charts, and suggested follow-ups.

🔹 **Artifacts:** a user can **Save** a chart or table from a response as an Artifact — a persistent, shareable snapshot, ideal for recurring analyses. Sharing an artifact shares the *saved output*; if a colleague clicks **Explore** or re-runs the query, **their** role permissions still apply. An artifact is a snapshot, not a privilege escalation.

> **What to tell customers:** AI usage is fully visible in the same Account Usage views you already use for warehouse and storage cost — no hidden meters. Governance *is* your existing RBAC + masking; there's no separate AI security layer to bolt on. And every answer is traceable to a user and cites its source.

## Step 5: Provision SI-Only Business Users

🛠️ The agent is live and branded — now let business users in. Provision a dedicated user, restrict it with **`ALLOWED_INTERFACES`**, and grant it the `SI_USER` role (which already carries the agent + CoWork-object `USAGE` from the publish step above). Business users reach the agent through **CoWork only** — no worksheets, no SQL, no drivers.

🔹 `ALLOWED_INTERFACES` controls which entry points a user can authenticate through:

| Value | Grants access to | Example |
|---|---|---|
| `SNOWFLAKE_INTELLIGENCE` | Natural-language CoWork UI only | `ai.snowflake.com` |
| `SNOWSIGHT` | Web SQL interface | `app.snowflake.com` |
| `STREAMLIT` | Streamlit-in-Snowflake apps | Streamlit in Snowsight |
| `DRIVERS` | JDBC / ODBC / Python connector | SnowSQL, scripts |

📌 Interface restriction is **additive to RBAC**, not a replacement — the user still sees only the data their role grants + masking allow. In production, provision these users via **SSO / SAML** instead of passwords.

In [ ]:
%%sql -r create_si_user
USE ROLE ACCOUNTADMIN;
CREATE USER IF NOT EXISTS COWORK_ENTERPRISE_DEMO_BIZ_USER
  PASSWORD = 'ChangeMe_CoWorkDemo_2026!'
  MUST_CHANGE_PASSWORD = TRUE
  DEFAULT_ROLE = COWORK_ENTERPRISE_DEMO_SI_USER
  DEFAULT_WAREHOUSE = COWORK_ENTERPRISE_DEMO_WH
  COMMENT = 'CoWork Enterprise Demo - SI-only business user (demo; dropped in cleanup)';
ALTER USER COWORK_ENTERPRISE_DEMO_BIZ_USER SET ALLOWED_INTERFACES = ('SNOWFLAKE_INTELLIGENCE');
GRANT ROLE COWORK_ENTERPRISE_DEMO_SI_USER TO USER COWORK_ENTERPRISE_DEMO_BIZ_USER;
USE ROLE COWORK_ENTERPRISE_DEMO_ADMIN;
SELECT 'SI-only user COWORK_ENTERPRISE_DEMO_BIZ_USER created' AS STATUS;


### 📌 Validate SI-Only Access (manual)

A notebook can't authenticate as another user, so confirm in a browser (incognito):

- ✅ **`ai.snowflake.com`** → sign in as the user → sees the curated agent and can chat
- ❌ **`app.snowflake.com`** (Snowsight) → sign-in blocked (*access restricted to allowed interfaces*)
- ❌ **SnowSQL / driver** connect → authentication blocked

🔹 If the user logs in but sees an **empty screen**, the role is missing `USAGE` on the CoWork object (granted in the publish step above) — the most common go-live gap.

## Step 6: Verify & Operate

📌 Confirm the agent is published, then reuse the golden questions as a scheduled **health check** (track success rate + p95 latency). Continuous, post-go-live improvement is Notebook 07.

In [ ]:
%%sql -r verify_published
SHOW AGENTS IN SNOWFLAKE INTELLIGENCE SNOWFLAKE_INTELLIGENCE_OBJECT_DEFAULT;


## ✅ Notebook 05 Complete

### What You Just Built:
- Golden smoke tests across four question types
- Curated Agent GPA evaluation dataset `COWORK_ENTERPRISE_DEMO.AGENTS.DEMO_AGENT_EVALSET` + config stage, with the scored-gate pattern
- Agent published to the shared CoWork object; consumer role granted object `USAGE`
- Agent + CoWork interface branding applied
- SI-only business user `COWORK_ENTERPRISE_DEMO_BIZ_USER` provisioned (`ALLOWED_INTERFACES = SNOWFLAKE_INTELLIGENCE`)

---

### Key Points:
- Evaluation is the go-live gate — publish only what clears the threshold.
- Publishing to the CoWork object is the curation gate; agents are invisible until added.
- The shared CoWork object is never created or dropped — we only ADD our agent.

---

### Next:

Continue to **Notebook 06 — Dev → Prod Promotion**: treat the agent spec as code and promote it through an evaluation gate. Then **Notebook 07** closes the continuous-improvement loop.